# MarketPulse: 03 - Baseline Forecasting Using sktime

This notebook implements and benchmarks core time-series baseline models using **`sktime`**.

### Models Investigated:
1. **Naive Last**: Predicts the most recent observed demand value ($y_{t+h} = y_t$).
2. **Naive Mean**: Predicts historical average demand ($\hat{y} = \bar{y}$).
3. **Seasonal Naive ($sp=7$)**: Repeats observation from the previous seasonal cycle ($y_{t+h} = y_{t+h-7}$).
4. **Exponential Smoothing**: Holt-Winters additive trend and seasonality model.
5. **Theta Forecaster**: Decomposition method (Assimakopoulos and Nikolopoulos, 2000).
6. **AutoARIMA**: Automatic order selection using AIC/BIC minimization.

### Evaluation Horizons:
- 7 days (short-term operational replenishment)
- 14 days (bi-weekly planning)
- 28 days (monthly M5 official horizon)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config
from src.data.hierarchy import M5Hierarchy
from src.forecasting.baselines import build_all_baseline_forecasters
from src.evaluation.benchmarking import BaselineBenchmarkEngine

sns.set_theme(style="whitegrid", palette="muted")
config = load_config(project_root / "configs" / "config.yaml")
df = pd.read_parquet(config.get_processed_path(config.data.processed_file))
print(f"Loaded {len(df):,} records.")

## 1. Preparing Benchmark Demand Series

We extract aggregate business demand (Level 1) and state/category demand to evaluate baseline behaviors.

In [ ]:
hierarchy = M5Hierarchy(metadata_df=df)
total_demand = hierarchy.aggregate_bottom_up(df, level=1)
y_total = total_demand.set_index("date")["sales"].asfreq("D").fillna(0.0)

plt.figure(figsize=(14, 4))
plt.plot(y_total.index, y_total.values, label="Total Demand Actuals", color="#2c3e50")
plt.title("Historical Aggregate Business Demand")
plt.xlabel("Date")
plt.ylabel("Units")
plt.legend()
plt.show()

## 2. Model Fitting & Forecasting on a 28-Day Horizon

In [ ]:
horizon = 28
y_train = y_total.iloc[:-horizon]
y_test = y_total.iloc[-horizon:]

forecasters = build_all_baseline_forecasters(sp=7)
results = {}

for name, model in forecasters.items():
    res = model.fit_and_evaluate(y_train=y_train, y_test=y_test, series_id="Total_Demand", sp=7)
    results[name] = res
    print(f"Model: {res.model_name:<25} | WAPE: {res.metrics['WAPE']:>6.2f}% | RMSE: {res.metrics['RMSE']:>6.2f} | Fit: {res.fit_time_sec:.4f}s")

## 3. Visualizing Model Forecasts vs Ground Truth

In [ ]:
plt.figure(figsize=(15, 6))
# Plot last 30 days of train
plt.plot(y_train.iloc[-30:].index, y_train.iloc[-30:].values, label="Train (Recent)", color="black", lw=1.5)
# Plot test actuals
plt.plot(y_test.index, y_test.values, label="Ground Truth Actuals", color="red", lw=2.5, linestyle="--")

colors = ["#3498db", "#9b59b6", "#2ecc71", "#e67e22", "#1abc9c", "#e74c3c"]
for (name, res), color in zip(results.items(), colors):
    plt.plot(y_test.index, res.predictions, label=f"{res.model_name} (WAPE: {res.metrics['WAPE']:.1f}%)", color=color, alpha=0.85)

plt.title(f"28-Day Forecast Horizon Comparison vs Actual Demand")
plt.xlabel("Date")
plt.ylabel("Units")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 4. Multi-Horizon Benchmarking across 7, 14, and 28 Days

In [ ]:
engine = BaselineBenchmarkEngine(forecasters=forecasters, horizons=[7, 14, 28], sp=7)
engine.evaluate_series(y_total, series_id="Total_Demand")
leaderboard = engine.get_leaderboard(sort_by="WAPE")
display(leaderboard)